<a href="https://colab.research.google.com/github/christianeiken-coder/Ai-Medical-Physics/blob/main/week06/pfml_w06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Lecture 06 Exercise: Data and Data Preprocessing

This exercise has **9 tasks**, worth **11 points**.

You will work with the file `patients.csv` provided alongside this notebook. You will prepare model-ready X and y from the raw data.

In case you want to, you can turn on Autosave in Colab, so your progress is saved. You can execute this, and any other cell by pressing Shift+Enter, or by pressing the little play button next to it.

In [3]:
%autosave 60

Autosaving every 60 seconds


### Core Ideas

- A dataset is a **table of observations**.
- Each **row** is one sample (one patient).
- Each **column** is one feature (a measured property).
- One column is the **label** (what we want to predict).
- The feature matrix **X** holds the input columns.
- The label vector **y** holds the target values.
- This course focuses on **supervised learning**.
- The label decides the task: classification or regression.

### Task 1 (1 point): Load and Inspect

Load `patients.csv` into a DataFrame named `patients` using `pd.read_csv`.

Then print:
- `patients.shape`
- `patients.info()`
- `patients.describe()`

Answer in a comment: which value in the data is a **sentinel missing marker** (i.e. a number that is *not* a real measurement)?

In [5]:
# TASK 1

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification

# Load the data
patients = pd.read_csv("patients.csv")


In [16]:
patients  #-9999 in der Spalte "counts"

,patient_id,age,dose_mgy,counts,outcome
0,1,34,2.5,543,0
1,2,51,3.1,602,1
2,3,29,NaN,487,0
3,4,63,4.8,711,1
4,5,44,2.9,-9999,0


In [6]:
patients.shape

(5, 5)

In [7]:
patients.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   patient_id  5 non-null      int64  
 1   age         5 non-null      int64  
 2   dose_mgy    4 non-null      float64
 3   counts      5 non-null      int64  
 4   outcome     5 non-null      int64  
dtypes: float64(1), int64(4)
memory usage: 332.0 bytes


In [8]:
patients.describe()

,patient_id,age,dose_mgy,counts,outcome
count,5.000000,5.000000,4.000000,5.000000,5.000000
mean,3.000000,44.200000,3.325000,-1531.200000,0.400000
std,1.581139,13.553597,1.014479,4734.371046,0.547723
min,1.000000,29.000000,2.500000,-9999.000000,0.000000
25%,2.000000,34.000000,2.800000,487.000000,0.000000
50%,3.000000,44.000000,3.000000,543.000000,0.000000
75%,4.000000,51.000000,3.525000,602.000000,1.000000
max,5.000000,63.000000,4.800000,711.000000,1.000000


### Task 2 (1 point): Define X and y

Set `feature_names = ["age", "dose_mgy", "counts"]`.

Create:
- `X` – a DataFrame with the three feature columns (use **two** pairs of brackets).
- `y` – a Series with the `outcome` column (use **one** pair of brackets).

Print `X.shape` and `y.shape`. They should be `(5, 3)` and `(5,)`.

Remember: keep them aligned so row *i* of X matches row *i* of y. Split them only **after** cleaning.

In [9]:
# TASK 2
feature_names = ["age", "dose_mgy", "counts"]

#bei data frames: df['col'] für spalten
X = patients[feature_names]
y = patients["outcome"]

In [10]:
X.shape

(5, 3)

In [12]:
y.shape

(5,)

### Task 3 (1 point): Detect Missing Values

Create a **copy** named `patients_clean`.

The `counts` column contains the sentinel value **-9999**, which is *not* a real measurement. Replace it with `np.nan` **before** any arithmetic.

Then print:
- `patients_clean.isna().sum()`
- `patients_clean.isna().mean() * 100`  (percentage missing per column)

Expected: `dose_mgy` has 1 missing value (already NaN), `counts` has 1 missing value (the former -9999).

In a comment, explain **why** the sentinel must be converted to NaN before computing statistics.

In [13]:
# TASK 3

'''NaN steht für not a number, es fehlt an der stelle also ein wert.
 manche programme können kein leeres feld speichern deshlab trägt man eine unmögliche zahl ein wie -9999.
mit dem unmöglichen wert würde natürlich was komplett falsches rauskommen weshalb man mit np.nan den wert
rausnimmt'''

patients_clean = patients.copy()
patients_clean["counts"] = patients_clean["counts"].replace(-9999, np.nan)

#isna() steht für is not available und gibt true oder false aus, prüft also jeden wert auf nan

In [14]:
patients_clean.isna().sum()     #mit sum() erhält man anzahl der fehlenden werte pro spalte

,0
patient_id,0
age,0
dose_mgy,1
counts,1
outcome,0


In [15]:
patients_clean.isna().mean() * 100

,0
patient_id,0.0
age,0.0
dose_mgy,20.0
counts,20.0
outcome,0.0


### Task 4 (1 point): Missingness Flags and Imputation

Two ways to handle missing data: **drop** (remove rows/columns) or **impute** (fill with a plausible value).

For this exercise, **impute** using the column **median** (safer with outliers than the mean).

Steps:
1. Add a boolean column `dose_missing` = `patients_clean["dose_mgy"].isna()` **before** filling.
2. Add a boolean column `counts_missing` = `patients_clean["counts"].isna()` **before** filling.
3. Fill `dose_mgy` with its median.
4. Fill `counts` with its median.
5. Verify: `patients_clean.isna().sum().sum()` should be 0.

The fact that a value was missing can itself be informative – this is called **missingness as a feature**.

In [ ]:
# TASK 4


### Task 5 (1 point): Scaling

Features live on very different scales (age: 0–100, dose: 0–5, counts: 0–10000). Distance-based methods and gradient descent struggle with uneven scales.

**Min-max normalisation** rescales to $[0, 1]$:

$$x' = \frac{x - \min(x)}{\max(x) - \min(x)}$$

**Standardisation (z-score)** centres at 0 and scales by the standard deviation:

$$z = \frac{x - \mu}{\sigma}$$

Compute both scaled versions of the `age` column using pandas (on `patients_clean`). Print the first 5 values of each.

In practice we use sklearn scalers on the full feature matrix X (you will do that in Task 6).

In [ ]:
# TASK 5


### Task 6 (1 point): Split First, Fit Later

The **right order** is:
1. Load data
2. **Split** into train / test
3. **Fit** the scaler on the training set **only**
4. **Transform** both sets with the training statistics

Steps:
- Re-create `X` (three numeric features) and `y` from `patients_clean`.
- Split with `train_test_split(X, y, test_size=0.2, random_state=42)`.
- Fit a `StandardScaler` on `X_train` **only**.
- Transform `X_train` and `X_test`.
- Print the shapes. Expected: X_train `(4, 3)`, X_test `(1, 3)`, y_train `(4,)`, y_test `(1,)`.

Never look at X_test while developing.

In [ ]:
# TASK 6


### Task 7 (1 point): Find the Leakage

**Data leakage** means information from the future or from the answer sneaks into the pipeline. Symptoms: too-good scores, then failure on new data.

Consider three pipelines:

| Pipeline | Description |
|----------|-------------|
| A | Fit the scaler on the **entire** dataset, **then** split into train/test. |
| B | Split into train/test **first**, then fit the scaler on `X_train` only. |
| C | Include the `outcome` column inside the feature matrix X. |

In comments below, answer:
1. Which pipelines leak information? Why?
2. Why does the leaked information make test scores **optimistic**?
3. Is leakage a model bug or a pipeline bug?

In [ ]:
# TASK 7
# Write your answers as comments here.


### Task 8 (2 points): Synthetic Data

When real data is small, private, or not yet collected, **synthetic data** (generated from a known rule) is useful for teaching, testing, and debugging.

Steps:
1. Generate a dataset with `make_classification(n_samples=200, n_features=5, random_state=0)`. Store as `X_syn`, `y_syn`.
2. Print their shapes. Expected: `(200, 5)` and `(200,)`.
3. Generate a second dataset with `random_state=1`. Store as `X_syn2`, `y_syn2`.
4. Check whether the two datasets are identical (use `np.array_equal`). They should **not** be.
5. In a comment, explain what `random_state` does and why it matters for reproducibility.

In [ ]:
# TASK 8


### Task 9 (2 points): Fix the Preprocessing Pipeline

The function `prepare_buggy` below has **exactly two bugs**:

1. The sentinel value `-9999` in `counts` is **not** converted to `np.nan` before scaling.
2. The scaler is **fitted before** the train/test split (data leakage).

Write a corrected function `prepare_fixed(patients)` that:
- Replaces `-9999` with `np.nan` in the `counts` column.
- Uses the three numeric features `age`, `dose_mgy`, `counts` as X.
- Splits into train/test **before** fitting the scaler.
- Fits `StandardScaler` on `X_train` only.
- Transforms both `X_train` and `X_test`.
- Returns `(X_train, X_test, y_train, y_test)` as NumPy arrays.

Expected shapes: X_train `(4, 3)`, X_test `(1, 3)`, y_train `(4,)`, y_test `(1,)`.

In [ ]:
def prepare_buggy(patients):
    X = patients[['age', 'dose_mgy', 'counts']]
    y = patients['outcome']
    scaler = StandardScaler().fit(X)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    X_train = scaler.transform(X_train)
    X_test = scaler.transform(X_test)
    return X_train, X_test, y_train, y_test

In [ ]:
# TASK 9

def prepare_fixed(patients):
    pass  # your code here


In [ ]:
# Quick test (uncomment after implementing prepare_fixed)
# patients_test = pd.read_csv('patients.csv')
# X_tr, X_te, y_tr, y_te = prepare_fixed(patients_test)
# print(X_tr.shape, X_te.shape, y_tr.shape, y_te.shape)
# Expected: (4, 3) (1, 3) (4,) (1,)

### Checklist: Raw File to X and y

- [ ] Load the file and check the shape.
- [ ] Inspect dtypes, missing values, and ranges.
- [ ] Find sentinel values like -9999.
- [ ] Split into train and test **before** transforming.
- [ ] Impute and scale using **training statistics only**.
- [ ] Return aligned X and y for training.

### Summary

- Datasets are tables: samples are rows, features are columns.
- The label is the column we predict: X are features, y is the label.
- pandas loads and inspects tabular data.
- Missing values must be found and handled **on purpose**.
- Scaling makes features comparable; fit it on **training data only**.
- Leakage gives optimistic scores and real-world failure.

**Don't forget to commit your solutions to GitHub!**